In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Extreme Model Triage Benchmark Cascade (`models/multi_extreme.ipynb`)

This notebook benchmarks the **2-Layer Extreme Triage Cascade Architecture** based on `TODO.md` structure using **13 Clinical Feature Engineered Inputs**:

### Decision Logic & Sequential Order
1. **Layer 1 (Sequential Extreme LR Regressors - `deploy/lr_extreme_model.rds`)**:
   - **1st Inference (ESI 1 Regressor)**: `if (ESI 1 Regressor output == "1")` -> Predict **ESI 1**.
   - **2nd Inference (ESI 5 Regressor)**: `else if (ESI 5 Regressor output == "5")` -> Predict **ESI 5**.
2. **Layer 2 (Intermediate RF Model - `deploy/rf_extreme_model.rds`)**:
   - `else` (`"neither"`) -> Predict Random Forest output (**ESI 2**, **ESI 3**, or **ESI 4**).

### Evaluation Protocol
- Evaluated on the **Complete Case Test Distribution Set** (15% Test Split from `.RData`).
- Uses **13 Clinical Feature Engineered Inputs** matching `lr_extreme.ipynb` and `rf_extreme.ipynb`.
- Reports **5x5 Confusion Matrix**, **Accuracy**, **Precision**, **Recall (Sensitivity)**, **PR-AUC**, **Multi-Class ROC-AUC**, and **Target Class Count Comparison Table**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(randomForest)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Complete .RData Dataset & Construct 13 Clinical Feature Engineered Inputs
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Construct 13 Clinical Feature Engineered Inputs
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))

# Strict Complete Case Analysis
df <- na.omit(df_feng)

cat(sprintf("Feature Engineered Test Population Ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("5-Class ESI Distribution:\n")
print(table(df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Extract Complete Case Test Partition (15% Test Set)
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
in_train_val <- createDataPartition(df$raw_esi, p = 1 - test_size, list = FALSE)
test_df      <- df[-in_train_val, ]

cat(sprintf("Feature Engineered Test Partition: %d rows\n", nrow(test_df)))
cat("Test Set Class Distribution:\n")
print(table(test_df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Read Pre-Trained Model Artifacts (Layer 1 LR Extreme & Layer 2 RF Extreme ESI 234)
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"

path_lr_ext  <- file.path(deploy_dir, "lr_extreme_model.rds")
path_esi1_lr <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
path_esi5_lr <- file.path(deploy_dir, "lr_feng_esi5_extreme_model.rds")
path_rf_ext  <- file.path(deploy_dir, "rf_extreme_model.rds")

cat("Reading pre-trained model artifacts from:", deploy_dir, "...\n")
mod_lr_ext  <- if (file.exists(path_lr_ext))  readRDS(path_lr_ext)  else NULL
mod_esi1_lr <- if (file.exists(path_esi1_lr)) readRDS(path_esi1_lr) else NULL
mod_esi5_lr <- if (file.exists(path_esi5_lr)) readRDS(path_esi5_lr) else NULL
mod_rf_ext  <- if (file.exists(path_rf_ext))  readRDS(path_rf_ext)  else NULL

cat(sprintf("Loaded Artifact Status:\n  - Layer 1 Dual Extreme LR Model (lr_extreme_model.rds): %s\n  - Layer 1 ESI 1 Model (lr_feng_esi1_extreme_model.rds): %s\n  - Layer 1 ESI 5 Model (lr_feng_esi5_extreme_model.rds): %s\n  - Layer 2 RF ESI 2,3,4 Model (rf_extreme_model.rds): %s\n",
            !is.null(mod_lr_ext), !is.null(mod_esi1_lr), !is.null(mod_esi5_lr), !is.null(mod_rf_ext)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Extreme Cascade Decision Engine & Sequential Prediction Logic
# ---------------------------------------------------------
predict_extreme_cascade <- function(data, mod_l1, mod_l2, mod_esi1 = NULL, mod_esi5 = NULL) {
  N <- nrow(data)
  
  # Extract Model 1 (ESI 1) and Model 5 (ESI 5)
  m1 <- if (!is.null(mod_esi1)) mod_esi1$model else (if (!is.null(mod_l1$model_esi1)) mod_l1$model_esi1 else mod_l1$model)
  m5 <- if (!is.null(mod_esi5)) mod_esi5$model else (if (!is.null(mod_l1$model_esi5)) mod_l1$model_esi5 else NULL)
  
  preproc_l1 <- if (!is.null(mod_l1$preproc)) mod_l1$preproc else (if (!is.null(mod_esi1$preproc)) mod_esi1$preproc else NULL)
  
  # 1. Preprocess for Layer 1 Models
  d_l1 <- data
  if (!is.null(preproc_l1)) {
    num_cols_l1 <- names(d_l1)[sapply(d_l1, is.numeric)]
    num_cols_l1 <- intersect(num_cols_l1, names(preproc_l1$mean))
    if (length(num_cols_l1) > 0) {
      d_l1[, num_cols_l1] <- predict(preproc_l1, d_l1[, num_cols_l1, drop = FALSE])
    }
  }
  
  # 2. Preprocess for Layer 2 Model (RF ESI 2,3,4)
  d_l2 <- data
  if (!is.null(mod_l2$preproc)) {
    num_cols_l2 <- names(d_l2)[sapply(d_l2, is.numeric)]
    num_cols_l2 <- intersect(num_cols_l2, names(mod_l2$preproc$mean))
    if (length(num_cols_l2) > 0) {
      d_l2[, num_cols_l2] <- predict(mod_l2$preproc, d_l2[, num_cols_l2, drop = FALSE])
    }
  }
  
  # 3. Layer 1 Predictions: ESI 1 FIRST -> THEN ESI 5
  prob1 <- predict(m1, newdata = d_l1, type = "probs")
  pred1 <- ifelse(is.matrix(prob1), colnames(prob1)[max.col(prob1, ties.method = "first")],
                  ifelse(prob1 >= 0.5, "1", "not_1"))
  
  if (!is.null(m5)) {
    prob5 <- predict(m5, newdata = d_l1, type = "probs")
    pred5 <- ifelse(is.matrix(prob5), colnames(prob5)[max.col(prob5, ties.method = "first")],
                    ifelse(prob5 >= 0.5, "5", "not_5"))
  } else {
    pred5 <- ifelse(pred1 == "5", "5", "not_5")
    prob5 <- prob1
  }
  
  # 4. Layer 2 RF Predictions ('2', '3', '4')
  d_rf_l2 <- d_l2[, setdiff(names(d_l2), c("raw_esi", "target_esi234", "target_layer1"))]
  d_rf_l2 <- na.roughfix(d_rf_l2)
  
  pred_l2 <- as.character(predict(mod_l2$model, newdata = d_rf_l2, type = "response"))
  prob_l2 <- predict(mod_l2$model, newdata = d_rf_l2, type = "prob")
  
  # Sequential Cascade Decision Rule: 1st Inference ESI 1 -> 2nd Inference ESI 5 -> 3rd Inference RF ESI 2, 3, 4
  pred_classes <- character(N)
  for (i in 1:N) {
    if (pred1[i] == "1") {
      pred_classes[i] <- "1"
    } else if (pred5[i] == "5") {
      pred_classes[i] <- "5"
    } else {
      pred_classes[i] <- pred_l2[i]  # "2", "3", or "4"
    }
  }
  
  # Construct 5-Class Probabilities Matrix
  p1 <- if (is.matrix(prob1) && "1" %in% colnames(prob1)) prob1[, "1"] else (if (is.numeric(prob1)) prob1 else rep(0, N))
  p5 <- if (is.matrix(prob5) && "5" %in% colnames(prob5)) prob5[, "5"] else (if (is.numeric(prob5)) prob5 else rep(0, N))
  
  p2 <- if (is.matrix(prob_l2) && "2" %in% colnames(prob_l2)) prob_l2[, "2"] else rep(0, N)
  p3 <- if (is.matrix(prob_l2) && "3" %in% colnames(prob_l2)) prob_l2[, "3"] else rep(0, N)
  p4 <- if (is.matrix(prob_l2) && "4" %in% colnames(prob_l2)) prob_l2[, "4"] else rep(0, N)
  
  prob_matrix <- cbind("1" = p1, "2" = p2, "3" = p3, "4" = p4, "5" = p5)
  pred_factor <- factor(pred_classes, levels = c("1", "2", "3", "4", "5"))
  return(list(pred_factor = pred_factor, prob_matrix = prob_matrix))
}

cat("Extreme Cascade Decision Engine Initialized!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Benchmark Extreme Cascade (Layer 1 Dual LR Extreme + Layer 2 RF ESI 234)
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
res_test <- predict_extreme_cascade(test_df, mod_lr_ext, mod_rf_ext, mod_esi1_lr, mod_esi5_lr)
pred_test <- res_test$pred_factor
prob_test <- res_test$prob_matrix
actual_test <- factor(test_df$raw_esi, levels = c("1", "2", "3", "4", "5"))

cm  <- confusionMatrix(pred_test, actual_test)
acc <- as.numeric(cm$overall["Accuracy"])

prec_by_class <- cm$byClass[, "Pos Pred Value"]
rec_by_class  <- cm$byClass[, "Sensitivity"]
macro_prec    <- mean(prec_by_class, na.rm = TRUE)
macro_rec     <- mean(rec_by_class,  na.rm = TRUE)

pr_auc_by_class <- numeric(5)
names(pr_auc_by_class) <- c("1", "2", "3", "4", "5")
for (cls in c("1", "2", "3", "4", "5")) {
  act_bin <- ifelse(actual_test == cls, 1, 0)
  pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_test[, cls])
}
macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)

roc_obj <- pROC::multiclass.roc(actual_test, prob_test)
macro_roc_auc <- as.numeric(roc_obj$auc)

actual_table <- table(actual_test)
pred_table   <- table(pred_test)
diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))

cat(sprintf("============================================================\n"))
cat(sprintf("   EXTREME CASCADE SYSTEM (DUAL LR EXTREME + RF ESI 234) BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", macro_roc_auc))
cat(sprintf("============================================================\n\n"))

cat("Per-Class Target Count & Performance Summary Table:\n")
per_class_metrics <- data.frame(
  Class        = c("1", "2", "3", "4", "5"),
  Actual_Count = as.numeric(actual_table),
  Pred_Count   = as.numeric(pred_table),
  Diff         = diff_str,
  Precision    = round(prec_by_class, 4),
  Recall       = round(rec_by_class, 4),
  PR_AUC       = round(pr_auc_by_class, 4)
)
print(per_class_metrics)

cat("\nFull 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
print(cm$table)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7A: Diagnostic Plot 1 - Per-Class Metrics Bar Chart
# ---------------------------------------------------------
metrics_long <- per_class_metrics %>%
  pivot_longer(cols = c("Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")

p_bar <- ggplot(metrics_long, aes(x = Class, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Precision" = "#2b5c8f", "Recall" = "#e07a5f", "PR_AUC" = "#81b29a")) +
  labs(title = "Extreme Cascade Model: Per-Class Performance (Test Set)",
       subtitle = "Comparing Precision, Recall, and PR-AUC across all 5 ESI levels",
       x = "ESI Level", y = "Metric Value") +
  theme(plot.title = element_text(face = "bold", size = 14), legend.position = "top")

if (!dir.exists("../plots")) dir.create("../plots", recursive = TRUE)
ggsave("../plots/multi_extreme_metrics_barchart.png", plot = p_bar, width = 9, height = 5, dpi = 300)
cat("Per-Class Metrics Bar Chart saved to: plots/multi_extreme_metrics_barchart.png\n")

p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7B: Diagnostic Plot 2 - 5-Class Overlaid Precision-Recall (PR) Curves
# ---------------------------------------------------------
pr_df_list <- list()
for (cls in c("1", "2", "3", "4", "5")) {
  prob_pos <- prob_test[, cls]
  ord <- order(prob_pos, decreasing = TRUE)
  act_sorted <- (actual_test[ord] == cls)
  tp <- cumsum(act_sorted)
  fp <- cumsum(!act_sorted)
  n_pos <- sum(act_sorted)
  rec  <- c(0, tp / n_pos)
  prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
  pr_df_list[[cls]] <- data.frame(
    Recall = rec,
    Precision = prec,
    Class = sprintf("ESI %s (PR-AUC = %.3f)", cls, pr_auc_by_class[cls])
  )
}

df_pr_all <- do.call(rbind, pr_df_list)

p_pr <- ggplot(df_pr_all, aes(x = Recall, y = Precision, color = Class)) +
  geom_line(size = 1.2) +
  theme_minimal() +
  scale_color_manual(values = c("#d90429", "#f77f00", "#2a9d8f", "#457b9d", "#1d3557")) +
  labs(title = "Extreme Cascade Model: Precision-Recall Curves (Complete Test Set)",
       subtitle = "Overlaid PR curves for all 5 ESI triage levels",
       x = "Recall (Sensitivity)", y = "Precision (Positive Predictive Value)") +
  theme(plot.title = element_text(face = "bold", size = 14), legend.position = "bottom")

ggsave("../plots/multi_extreme_pr_curves.png", plot = p_pr, width = 8, height = 5.5, dpi = 300)
cat("PR Curves plot saved to: plots/multi_extreme_pr_curves.png\n")

p_pr

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7C: Diagnostic Plot 3 - 5-Class Overlaid ROC Curves
# ---------------------------------------------------------
roc_df_list <- list()
for (cls in c("1", "2", "3", "4", "5")) {
  r_obj <- pROC::roc(actual_test == cls, prob_test[, cls], quiet = TRUE)
  roc_df_list[[cls]] <- data.frame(
    FPR = 1 - r_obj$specificities,
    TPR = r_obj$sensitivities,
    Class = sprintf("ESI %s (AUC = %.3f)", cls, r_obj$auc)
  )
}

df_roc_all <- do.call(rbind, roc_df_list)

p_roc <- ggplot(df_roc_all, aes(x = FPR, y = TPR, color = Class)) +
  geom_line(size = 1.2) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "gray50") +
  theme_minimal() +
  scale_color_manual(values = c("#d90429", "#f77f00", "#2a9d8f", "#457b9d", "#1d3557")) +
  labs(title = "Extreme Cascade Model: ROC Curves Comparison (Complete Test Set)",
       subtitle = "Overlaid 1-vs-Rest ROC curves for all 5 ESI triage levels",
       x = "False Positive Rate (1 - Specificity)", y = "True Positive Rate (Sensitivity)") +
  theme(plot.title = element_text(face = "bold", size = 14), legend.position = "bottom")

ggsave("../plots/multi_extreme_roc_curves.png", plot = p_roc, width = 8, height = 5.5, dpi = 300)
cat("ROC Curves plot saved to: plots/multi_extreme_roc_curves.png\n")

p_roc